In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd

from bs4 import BeautifulSoup

from time import sleep

from datetime import datetime

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

import datetime

import os

from selenium.webdriver.chrome.service import Service as ChromeService

import re


# %%

In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'BA KOMVP'

print(f"Running {regulatorName} Web Scraping Tool v.1.0")


now=datetime.datetime.now()

filename= 'BA KOMVP Data {}.xlsx'.format(str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') 

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


Running BA KOMVP Web Scraping Tool v.1.0


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\BA KOMVP'

In [ ]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

In [ ]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

def scrollinAndClick(xpath,key_press=False):
    if len(xpath) != 0 :

        for times in range(60):

            try:
                driver.find_element(By.XPATH, xpath).click()

                sleep(5)

                break
            except:
                # print(f"[ERROR] : trying {times+1}/10 to key press 'DOWN' (scrolling)")
                sleep(5)
                if key_press:                    

                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)
        else:   
            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')

        

def find_zip_code(string):
    match = re.search(r'\d{5}', string)
    if match:
        return match.group()
    else:
        return None

In [ ]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------


regdict={'BA KOMVYP 1': 'https://www.komvp.gov.ba/en/market-participants/issuers', 
         'BA KOMVYP 2': 'https://www.komvp.gov.ba/en/market-participants/fmc' ,
         'BA KOMVYP 3': 'https://www.komvp.gov.ba/en/market-participants/funds',
        }


Typology={
        'BA KOMVYP 1': 'List of Issuers', 
        'BA KOMVYP 2': 'List of Fund Management Companies' ,
        'BA KOMVYP 3': 'List of Investment Funds',
         }
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')
		  

In [ ]:
# %%
#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    inner_links = []
    driver.delete_all_cookies()
    inner_links = []
    print('Working with {}.'.format(reg))
    driver.get(regdict[reg])
    sleep(1)
    soup=BeautifulSoup(driver.page_source, "html.parser")
    page = soup.find('div', id='DataTables_Table_0_paginate')
    next = page.find('li',id = 'DataTables_Table_0_next')
    while next:
        soup2=BeautifulSoup(driver.page_source, "html.parser")
        page = soup2.find('div', id='DataTables_Table_0_paginate')
        next = page.find('li',id = 'DataTables_Table_0_next')
        tbody = soup2.find('tbody')
        trs = tbody.find_all('tr')
        for tr in trs:
            internal_id = tr.find_all('td')[0].text
            inner_link = 'https://www.komvp.gov.ba'+tr.find_all('td')[-1].find('a')['href']
            inner_links.append(inner_link)
        
        scrollinAndClick('//*[@id="DataTables_Table_0_next"]')
        #print(f'-- Current Page -- ' + page.find('li',id = 'DataTables_Table_0_next').previous_element)
        if 'disabled' in next['class']:
            print('Next Button is unavaliable')
            next = False
            
    for index,detail_info in enumerate(inner_links):

        driver.get(detail_info)
        driver.delete_all_cookies()
        sleep(1)
        detail_soup = BeautifulSoup(driver.page_source, "html.parser")
        try:
            trs = detail_soup.find('tbody').find_all('tr')
            print('Current Index:' + str(index+1))
        except:
            driver.refresh()
            trs = detail_soup.find('tbody').find_all('tr')
            print('Current Index:' + str(index+1))
        if reg == 'BA KOMVYP 1':            
            for tr in trs:
                if 'Company Name' in tr.text and 'Short Company Name' not in  tr.text:
                    name =tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(name)
                    sqldict['Name'].append(name)
                    sqldict['InternalID_1_type'].append('ID')
                    sqldict['InternalID_1'].append(str(detail_info.split('/')[-1]))
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    print(str(detail_info.split('/')[-1]))

                elif 'Address' == tr.find_all('td')[0].text.strip() :
                    address =tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(address)
                    sqldict['Address_1'].append(address)
                    #sqldict['ListProcessDate'].append(processdate)
                    print(tr.find_all('td'))
                elif 'Phone' in tr.text:
                    phone = tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(phone)
                    sqldict['Phone'].append(phone)
                elif 'Fax' in tr.text:
                    fax = tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(fax)
                    sqldict['Fax'].append(fax)
                elif 'Company Identification Number' in tr.text:
                    id2 = tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(id2)
                    sqldict['InternalID_2_type'].append('Company Identification Number')
                    sqldict['InternalID_2'].append(str(id2))
                elif 'E-mail' in tr.text:
                    email = tr.find_all('td')[-1].text.strip()
                    sqldict['Email'].append(email)
                    print(email)
            sqldict = bourange_same_length_array(sqldict)
        elif reg == 'BA KOMVYP 2' or  reg == 'BA KOMVYP 3': 
            for tr in trs:
                if 'Company Name' in tr.text and 'Short Company Name' not in  tr.text:
                    name =tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(name)
                    sqldict['Name'].append(name)
                    sqldict['InternalID_1_type'].append('ID')
                    sqldict['InternalID_1'].append(str(detail_info.split('/')[-1]))
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegulationType'].append('Regulated')
                    print(str(detail_info.split('/')[-1]))
                
                elif 'Address' == tr.find_all('td')[0].text.strip() :
                    address =tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(address)
                    sqldict['Address_1'].append(address)
                    #sqldict['ListProcessDate'].append(processdate)
                    print(tr.find_all('td'))
                elif 'Phone' in tr.text:
                    phone = tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(phone)
                    sqldict['Phone'].append(phone)
                elif 'Fax' in tr.text:
                    fax = tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(fax)
                    sqldict['Fax'].append(fax)
                elif 'Company Identification Number' in tr.text:
                    id2 = tr.find_all('td')[-1].text.strip() if tr.find_all('td')[-1].text.strip() else ''
                    print(id2)
                    sqldict['InternalID_2_type'].append('Company Identification Number')
                    sqldict['InternalID_2'].append(str(id2))
                elif 'E-mail' in tr.text:
                    email = tr.find_all('td')[-1].text.strip()
                    sqldict['Email'].append(email)
                    print(email)
                elif 'Web page' in tr.text:
                    web = tr.find_all('td')[-1].text.strip()
                    sqldict['Website'].append(web)
                    print(web)
                elif 'License Issuance' in tr.text  and 'Date License Issuance' not in  tr.text:
                    License = tr.find_all('td')[-1].text.strip()
                    sqldict['RegulationTypeCode'].append(str(License))
                    print(License)
                elif 'Date License Issuance' in tr.text:
                    Date_License = tr.find_all('td')[-1].text.strip()
                    sqldict['RegulationDate'].append(str(Date_License))
                    print(Date_License)
                elif 'Court Registry File Number' in tr.text:
                    court_num = tr.find_all('td')[-1].text.strip()
                    sqldict['InternalID_3'].append(str(court_num))
                    sqldict['InternalID_3_type'].append('Court Registry File Number')
                    print(court_num)                   
            sqldict = bourange_same_length_array(sqldict)    


Working with BA KOMVYP 2.
Next Button is unavaliable
Current Index:1
Društvo za upravljanje fondovima SAFE INVESTMENT d.o.o. Sarajevo
04-19
Danijela Ozme 1, Sarajevo
[<td>Address</td>, <td>Danijela Ozme 1, Sarajevo</td>]
033550120
033550121
65-01-0415-24
info@safeinvestment.ba
www. safeinvestment.ba
05/3-19-73/24
18.04.2024.
Current Index:2
Društvo za upravljanje fondovima SME INVEST d.o.o. Mostar
04-8
Kneza Branimira 2/II, Mostar
[<td>Address</td>, <td>Kneza Branimira 2/II, Mostar</td>]
036328687
36328677
4227003080004
iinvest@smeinvest.ba
www.sme-invest.ba
1-10288
03-19-105/00
31.08.2000.
Current Index:3
Društvo za upravljanje fondovima “ASA ASSET MANAGEMENT" d.o.o. Sarajevo
04-11
Bulevar Meše Selimovića 16, Sarajevo
[<td>Address</td>, <td>Bulevar Meše Selimovića 16, Sarajevo</td>]
033407170
033766996
4200052540007
blago@bih.net.ba
www.blago.ba
1-22743
05-19-160/00
08.11.2000.
Current Index:4
Društvo za upravljanje fondovima “BOSINVEST” d.o.o. Sarajevo - NEAKTIVNO
04-5
Vilsonovo šeta

In [ ]:

# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 36 values.
Key 'priority' has 36 values.
Key 'ListLabel' has 36 values.
Key 'Typology' has 36 values.
Key 'EntryType' has 36 values.
Key 'Name' has 36 values.
Key 'InternalID_1' has 36 values.
Key 'InternalID_1_type' has 36 values.
Key 'InternalID_2' has 36 values.
Key 'InternalID_2_type' has 36 values.
Key 'InternalID_3' has 36 values.
Key 'InternalID_3_type' has 36 values.
Key 'CoType' has 36 values.
Key 'License_Type' has 36 values.
Key 'Address_1' has 36 values.
Key 'Address_2' has 36 values.
Key 'City' has 36 values.
Key 'Zip' has 36 values.
Key 'Cntry' has 36 values.
Key 'Phone' has 36 values.
Key 'Fax' has 36 values.
Key 'Website' has 36 values.
Key 'Email' has 36 values.
Key 'RegulationType' has 36 values.
Key 'RegulationTypeCode' has 36 values.
Key 'RegulationDate' has 36 values.
Key 'CancellationDate' has 36 values.
Key 'RegCtry' has 36 values.
Key 'RegCode' has 36 values.
Key 'ListCode' has 36 values.
Key 'ListLanguage' has 36 values.
Key 'ListValidityDate' h

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_12880\3068940208.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('list_23.csv')